# 3. minio-to-azure

Muevo la tabla Iceberg del paso 2 (`taxis.yellow_tripdata`) hacia Azure, a la carpeta de nuestro grupo (GRUPO_3).

In [1]:
import dlt
from dlt.common.libs.pyiceberg import get_catalog

`get_catalog` arma la conexión al catálogo de Nessie leyendo directo la sección `iceberg_catalog` de secrets.toml, así no tengo que volver a escribir usuario ni clave acá. Con eso cargo la tabla y la paso a pyarrow.

In [2]:
catalog = get_catalog()

iceberg_table = catalog.load_table("taxis.yellow_tripdata")
arrow_table = iceberg_table.scan().to_arrow()
print("filas leídas:", arrow_table.num_rows)

filas leídas: 3475226


El resource solo entrega lo que ya leímos arriba.

In [3]:
@dlt.resource(name="yellow_tripdata", write_disposition="replace")
def yellow_tripdata_azure():
    yield arrow_table

bucket_url apunta directo a la carpeta de Azure que nos asignó el profe (GRUPO_3). Las credenciales de la cuenta de Azure están en el secrets, no acá.

In [4]:
pipeline = dlt.pipeline(
    pipeline_name="minio_to_azure",
    destination=dlt.destinations.filesystem(
        bucket_url="abfss://clase-4-dlt@fhbd.dfs.core.windows.net/GRUPO_3"
    ),
    dataset_name="taxis_iceberg",
)

Corremos pidiendo parquet como formato de salida.

In [5]:
load_info = pipeline.run(yellow_tripdata_azure, loader_file_format="parquet")
print(load_info)

Pipeline minio_to_azure load step finished in 1 minute and 2.83 seconds
1 load package(s) were loaded to destination filesystem and into dataset taxis_iceberg
The filesystem destination used abfss://clase-4-dlt@fhbd.dfs.core.windows.net/GRUPO_3 location to store data
Load package 1788806071.5603728 is LOADED and contains no failed jobs
